### Module import

In [7]:
# coding=utf-8
from __future__ import absolute_import, division, print_function

# =========================
# Standard library
# =========================
import os
import re
import time
import math
import copy
import glob
import json
import shutil
import random
import logging
import argparse
import datetime
import platform
import contextlib
from collections import defaultdict, Counter

# =========================
# Scientific / Data stack
# =========================
import numpy as np
import pandas as pd
import scipy
from scipy import signal

# =========================
# Imaging / Visualization
# =========================
from PIL import Image, ImageOps, ImageDraw
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.pyplot import figure
import cv2
from skimage import filters, metrics, transform
from skimage.metrics import (
    mean_squared_error as compare_MSE,
    structural_similarity as compare_SSIM,
    peak_signal_noise_ratio as compare_PSNR,
)

# =========================
# Progress / Debug
# =========================
from tqdm import tqdm
from IPython.core.debugger import set_trace as Tracer


# =========================
# PyTorch & TorchVision
# (no torch.utils.data per your request)
# =========================
import torch
import torch.nn.functional as F
import torch.nn.parallel
from torch import nn, optim
from torchvision import datasets, models, transforms
import torchvision.models as tv_models  # if you need explicit alias
from torchvision.transforms.functional import InterpolationMode
from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader
# =========================
# Training utilities
# =========================
from apex import amp
from apex.parallel import DistributedDataParallel as DDP
from torch.utils.tensorboard import SummaryWriter

# =========================
# Grad-CAM (both custom and library)
# =========================
from utils import visualize_cam, Normalize
from gradcam import GradCAM, GradCAMpp             # your local/custom grad-cam
from pytorch_grad_cam import GradCAMPlusPlus       # third-party grad-cam
from pytorch_grad_cam.utils.image import show_cam_on_image

# =========================
# ONNX / XGBoost conversion
# =========================
import onnx
import onnxruntime
from skl2onnx import to_onnx, update_registered_converter
from skl2onnx.common.shape_calculator import calculate_linear_classifier_output_shapes
from onnxmltools.convert import convert_xgboost
from onnxmltools.convert.common import data_types
from xgboost import XGBClassifier

# =========================
# Parallel / System utils
# =========================
import ray
from ray.util.multiprocessing import Pool
import py7zr
import psutil
import pickle

# =========================
# Runtime config
# =========================
Image.MAX_IMAGE_PIXELS = None  # Disable PIL decompression bomb check


In [2]:
# create a binary label 0 and 1 for patch-level original metadata, which will be used later to load data 
# with labels in data loader along with patch saliency scores

#conversion category to binary labels

  
labelBenign, labelMalignant, labelExclude = '0', '1', '2'
#Define labels used for normal/benign tissue
#'a': normal adipose.
#'s': normal stroma tissue excluding adipose.
#'o': other normal tissue including parenchyma, adenosis, lobules, blood vessels, etc.
labelsBenign = ['a', 's', 'o', 'normal']

#Define labels used for malignant tissue
#'d': IDC tumor
#'l': ILC tumor
#'ot': other tumor areas including DCIS, biopsy site, and slightly defocused tumor regions.
labelsMalignant = ['d', 'l', 'ot', 'tumor']

#Define labels used for tissues to be excluded
#'ft': defocused but still visually tumor-like areas.
#'f': severly out-of-focusing areas. 
#'b': background. 
#'e': bubbles.
labelsExclude = ['ft', 'f', 'b', 'e', 'exclude']


#Load/synchronize data labeling, drop excluded rows, and extract relevant metadata
def loadMetadata_patches(filename):
    metadata = pd.read_csv(filename, header=0, names=['Sample', 'Index', 'Row', 'Column', 'Label'], converters={'Sample':str,'Index':str, 'Row':int, 'Column':int, 'Label':str})
    metadata['Label'] = metadata['Label'].replace(labelsBenign, labelBenign)
    metadata['Label'] = metadata['Label'].replace(labelsMalignant, labelMalignant)
    metadata['Label'] = metadata['Label'].replace(labelsExclude, labelExclude)
    metadata = metadata.loc[metadata['Label'] != labelExclude]
    return [np.squeeze(data) for data in np.split(np.asarray(metadata), [1, 2, 4], -1)]



csv_dir = '/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/CSV_Files/metadata_patches.csv'
metadata = pd.read_csv(csv_dir)
# Create a mapping dictionary
label_map = {lbl: 0 for lbl in labelsBenign}
label_map.update({lbl: 1 for lbl in labelsMalignant})
label_map.update({lbl: None for lbl in labelsExclude})  # Exclude rows

# Apply mapping
metadata['Binary_Label'] = metadata['Label'].map(label_map)

# Drop excluded rows
metadata = metadata.dropna(subset=['Binary_Label'])

# Convert float → int
metadata['Binary_Label'] = metadata['Binary_Label'].astype(int)

#Output directory and file
output_dir = "/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/CSV_Files"

#This CSV will be used to save the gradcam+++ saliency scores as well and then it will be used later to load, img, label and saliency scores all together in main.py
output_csv = os.path.join(output_dir, "metadata_patches_with_grad_cam++_binary_label.csv")

# Save updated CSV
metadata.to_csv(output_csv, index=False)


### Load image

### preprocess image

In [5]:
#Class dataset for Whole slide images, will be used later for data loader to feed it to the dataloader
class WSIDataset(Dataset):
    def __init__(self, wsi_imgs_dir, wsi_label_csv, transform=None, wsi_names=None):
        """
        Args:
            wsi_imgs_dir (str): Directory containing WSI image files.
            wsi_label_csv (str): Path to the CSV file with WSI labels.
            transform (callable, optional): Transform to be applied to the images.
            wsi_names (list, optional): List of WSI names to include in the dataset.
                                         If None, all WSI names in the CSV are used.
        """
        self.wsi_imgs_dir = wsi_imgs_dir
        self.metadata = pd.read_csv(wsi_label_csv)
        self.transform = transform

        # Add `.jpg` extension to the sample names
        self.wsi_names_csv = {name + '.jpg' for name in self.metadata['Sample'].str.strip()}

        # Filter valid image files and apply fold-based filtering if `wsi_names` is provided
        if wsi_names:
            wsi_names_set = {name + '.jpg' for name in wsi_names}
            self.image_files = [file for file in os.listdir(wsi_imgs_dir)
                                if file.endswith('.jpg') and file in wsi_names_set]
        else:
            self.image_files = [file for file in os.listdir(wsi_imgs_dir)
                                if file.endswith('.jpg') and file in self.wsi_names_csv]

        # Create a mapping between image names and labels
        self.image_labels = {f"{row['Sample']}.jpg": row['Binary_Label'] for _, row in self.metadata.iterrows()}

    def __len__(self):
        """Returns the total number of samples."""
        return len(self.image_files)

    def __getitem__(self, idx):
        """
        Args:
            idx (int): Index of the sample.
        
        Returns:
            tuple: (image, label) where image is the transformed image and label is its label.
        """
        img_name = self.image_files[idx]
        img_path = os.path.join(self.wsi_imgs_dir, img_name)
        label = self.image_labels[img_name]

        # Load the image
        image = Image.open(img_path).convert('RGB')

        # Apply transforms if specified
        if self.transform:
            image = self.transform(image)

        return image, label, img_name



In [8]:

# Paths
wsi_imgs_dir = '/data/users4/pafshin1/My_Projects copy/Rands/DATA/PATCHES/INPUT_WSI/'
wsi_label_csv = '//data/users4/pafshin1/My_Projects/All_Labels/Updated_grad_cam_csv/wsi_level_labels_with_mixed.csv'

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224
    transforms.ToTensor(),          # Convert images to tensors
])


# fold_idx = 1  # Use the first fold as the test set
# train_dataset, val_dataset, test_dataset, train_folds, test_folds = create_datasets_for_fold(
#     wsi_imgs_dir, wsi_label_csv, manualFolds, fold_idx, transform=transform
# )

# Create DataLoaders
train_dataset = WSIDataset(wsi_imgs_dir, wsi_label_csv, transform=transform, wsi_names=None)
#depending on the gpu capacity, increase/decrease the batch size
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)



In [9]:
len(train_dataset)


66

### Feedforward image, calculate GradCAM/GradCAM++, and gather results

### Load Grad-Cam Model


In [10]:

weightsResNet = 'IMAGENET1K_V2'

#Which pre-trained weight sets should be used for DenseNet model: 'IMAGENET1K_V1'
weightsDenseNet = 'IMAGENET1K_V1'

# # saliency Maps
#Load pre-trained DenseNet
# Define the DenseNet model without loading ImageNet weights
model_DenseNet = models.densenet169(weights=weightsDenseNet)

#Replace the in-built classifier; unclear how this structure and these hyperparameters were determined
model_DenseNet.classifier = nn.Sequential(
    nn.Linear(model_DenseNet.classifier.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2)
)

# Load the weights from fine tune model over first batch WSI
checkpoint_path = '/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Densenet/Results/Densenet_1e-4_adam_1/best_overall_model.pth'
checkpoint = torch.load(checkpoint_path)
# Load the state_dict from the checkpoint
model_DenseNet.load_state_dict(checkpoint)

model_DenseNet = model_DenseNet.cuda()
_ = model_DenseNet.train(False)

#Create GradCAMPlusPlus model; see https://github.com/jacobgil/pytorch-grad-cam for additional models and options
Densenet_GradCamPlusPlus_model = GradCAMPlusPlus(model=model_DenseNet, target_layers=[model_DenseNet.features[-1]])

/data/users4/pafshin1/tmp/ipykernel_1331504/3621458942.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


In [11]:
model_DenseNet.load_state_dict(checkpoint)

<All keys matched successfully>

In [12]:

# Directory to save the saliency maps
Saliency_saved_dir = '/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/'


def Saliency_map_grad_cam(loader, Saliency_saved_dir, model_GradCamPlusPlus):
    """
    Args:
        loader: DataLoader containing WSI image files.
        Saliency_saved_dir: Path for saving the saliency maps of each WSI.
        model_GradCamPlusPlus: Grad-CAM++ model instance.
        
    Returns:
        saliencyMaps: Numpy array of all saliency maps.
        wsi_names: List of all processed WSI names.
    """
    # Create the save directory if it doesn't exist
    os.makedirs(Saliency_saved_dir, exist_ok=True)
    
    wsi_names = []
    saliencyMaps = []

    # Iterate over the data loader
    for data, _, wsi_name in tqdm(loader, total=len(loader), desc='Saliency Mapping', leave=True, ascii=False): 
        # Get the saliency map for the current batch
        saliency_map = model_GradCamPlusPlus(input_tensor=data, targets=None)
        
        # Ensure saliency_map is in the right format
        if not isinstance(saliency_map, np.ndarray):
            saliency_map = saliency_map.cpu().detach().numpy()  # Convert to NumPy if tensor

        # Save each saliency map for each image in the batch
        for i in range(len(wsi_name)):
            # Extract WSI name and create a filename
            wsi = wsi_name[i].split('.')[0]
            filename = f"saliencymap_{wsi}.npy"
            saliency_map_filename = os.path.join(Saliency_saved_dir, filename)
            
            # Save the saliency map
            np.save(saliency_map_filename, saliency_map[i])
            wsi_names.append(wsi)

            print(f"Saliency map saved for {wsi} to {saliency_map_filename}")
        
        # Append the current batch's saliency maps
        saliencyMaps.append(saliency_map)

        # Free GPU memory
        if hasattr(model_GradCamPlusPlus, 'outputs'):
            del model_GradCamPlusPlus.outputs
        torch.cuda.empty_cache()
    
    # Combine all saliency maps
    saliencyMaps = np.vstack(saliencyMaps)  # Ensure compatible shapes before stacking

    return saliencyMaps, wsi_names

saliencyMaps, wsi_names = Saliency_map_grad_cam(train_loader, Saliency_saved_dir, Densenet_GradCamPlusPlus_model)

Saliency Mapping:  20%|██        | 1/5 [02:31<10:07, 151.88s/it]

Saliency map saved for 10_3 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_10_3.npy
Saliency map saved for 11_3 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_11_3.npy
Saliency map saved for 12_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_12_1.npy
Saliency map saved for 13_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_13_1.npy
Saliency map saved for 14_2 to /data/users4/pafshin1/Implementation/Vision Trans

Saliency Mapping:  40%|████      | 2/5 [04:00<05:43, 114.64s/it]

Saliency map saved for 27_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_27_1.npy
Saliency map saved for 28_2 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_28_2.npy
Saliency map saved for 29_2 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_29_2.npy
Saliency map saved for 2_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_2_1.npy
Saliency map saved for 30_2 to /data/users4/pafshin1/Implementation/Vision Transfo

Saliency Mapping:  60%|██████    | 3/5 [05:01<03:00, 90.23s/it] 

Saliency map saved for 43_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_43_1.npy
Saliency map saved for 44_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_44_1.npy
Saliency map saved for 45_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_45_1.npy
Saliency map saved for 46_2 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_46_2.npy
Saliency map saved for 47_2 to /data/users4/pafshin1/Implementation/Vision Trans

Saliency Mapping:  80%|████████  | 4/5 [06:38<01:32, 92.74s/it]

Saliency map saved for 58_2 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_58_2.npy
Saliency map saved for 59_2 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_59_2.npy
Saliency map saved for 5_3 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_5_3.npy
Saliency map saved for 60_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_60_1.npy
Saliency map saved for 61_1 to /data/users4/pafshin1/Implementation/Vision Transfo

Saliency Mapping: 100%|██████████| 5/5 [06:47<00:00, 81.56s/it]

Saliency map saved for 8_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_8_1.npy
Saliency map saved for 9_3 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/saliencymap_9_3.npy


In [13]:

len(os.listdir('/data/users4/pafshin1/My_Projects copy/Rands/CODE/Saliency_Densenet/Saliency_Maps_batch1_full_dataset'))

66

In [14]:
def contiguousTensor(inputs):
    return torch.from_numpy(inputs).contiguous()
    
#Rescale tensor; issues with lambda functions when using multiprocessing
def rescaleTensor(inputs):
    return inputs.to(dtype=torch.get_default_dtype()).div(255)
    
#Generate a torch transform needed for preprocessing image data
def generateTransform(resizeSize=[], rescale=False, normalize=False):
    transform = [contiguousTensor]
    if len(resizeSize) > 0: transform.append(v2.Resize(tuple(resizeSize)))
    if rescale: transform.append(rescaleTensor)
    if normalize: transform.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    return transforms.Compose(transform)    

# Function to generate a Grad-CAM heatmap
def gen_cam(image, mask):
    # Create a heatmap from the Grad-CAM mask
    heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255

    # Superimpose the heatmap on the original image
    cam = (1 - 0.5) * heatmap + 0.5 * image
    cam = cam / np.max(cam)  # Normalize the result
    return np.uint8(255 * cam)  # Convert to 8-bit image

In [15]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm

import os
import numpy as np
import torch
import torch.nn.functional as F
from torchvision.utils import save_image
from tqdm import tqdm
from PIL import Image
import sys

wsi_images_dir = '/data/users4/pafshin1/My_Projects copy/Rands/DATA/PATCHES/INPUT_WSI'
output_dir = '/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/'
metadata_csv_dir = '/data/users4/pafshin1/My_Projects/All_Labels/Updated_grad_cam_csv/filtered_metadata_full_gradcam++.csv'
# Directory to save the saliency maps
Saliency_maps_dir = '/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/Saliency_Densenet/Saliency_Maps_batch1_full_dataset/'


patch_size = 400






In [16]:
metadata = pd.read_csv(metadata_csv_dir)
prefix="Densenet"



# Define transforms
transformation = transforms.Compose([
    transforms.ToTensor(),          # Convert images to tensors
])

# Directories
wsi_images_dir = '/data/users4/pafshin1/My_Projects copy/Rands/DATA/PATCHES/INPUT_WSI'
# save_dir = '/data/users4/pafshin1/Implementation/Grad-CAM_Tutorial2/gradcam_plus_plus-pytorch/Implementation/Grad-CAM_Tutorial2/gradcam_plus_plus-pytorch/outputs/Resulting_arrays'

# Ensure the result directory exists
os.makedirs(Saliency_maps_dir, exist_ok=True)

# Check if the WSI images directory exists
if not os.path.exists(wsi_images_dir):
    print(f"Directory '{wsi_images_dir}' does not exist. Please check the path.")
else:
    print(f"Directory '{wsi_images_dir}' found. Proceeding with loading saliency maps.")

# Process each WSI image by loading its corresponding saliency map
for wsi_name in tqdm(wsi_names, desc='Processing WSI Images', leave=True):
    wsi_name_without_extension = os.path.splitext(wsi_name)[0]
    print(f"Processing WSI: {wsi_name}")

    # File paths
    wsi_image_filename = os.path.join(wsi_images_dir, wsi_name + '.jpg')
    saliency_map_filename = os.path.join(Saliency_maps_dir, f"saliencymap_{wsi_name_without_extension}.npy")
    
    # Check if the WSI image file exists
    if os.path.exists(wsi_image_filename):
        # Load the WSI image
        wsi_image = Image.open(wsi_image_filename).convert('RGB')
        torch_img = transformation(wsi_image)  
        torch_img = torch_img.unsqueeze(0)

        # Check if the .npy file exists
        if os.path.exists(saliency_map_filename):
            # Load the saliency map
            saliency_map = np.load(saliency_map_filename)
             
            if saliency_map.ndim == 2:
                saliency_map = np.expand_dims(saliency_map, axis=0)
        
            # Pass only spatial dimensions (height, width) to Resize
            transform2 = generateTransform(resizeSize=torch_img.shape[2:], rescale=False, normalize=False)
            
            mask_pp_cpu, saliency_map_high_resolution  = transform2(saliency_map), transform2(saliency_map)[0].numpy() 
            mask_pp_cpu = mask_pp_cpu.unsqueeze(0) # 1, 1, height, width requires for Visualize_cam function 
            
            # Filter metadata for the current WSI
            # Define column names dynamically based on the prefix
            importance_column = f"{prefix}_Gradcam_Saliency_Importance"
            weight_column = f"{prefix}_Gradcam_Weight"

            # Initialize columns if not already present
            if importance_column not in metadata.columns:
                metadata[importance_column] = 0.0
            if weight_column not in metadata.columns:
                metadata[weight_column] = 0.0
                
            wsi_metadata = metadata[metadata['Sample'] == wsi_name]
            
            patchSize = 400
            
            # Iterate over each patch in the current WSI
            for _, row in wsi_metadata.iterrows():
                patch_index = row['Index']
                patch_row, patch_column = row['Row'], row['Column']
                
                
                # Extract the patch's saliency map
                patchSaliencyMap = saliency_map_high_resolution[patch_row:patch_row + patchSize, patch_column:patch_column + patchSize]
                
            
                # Check if the patchSaliencyMap is empty or has unexpected dimensions
                if patchSaliencyMap.size == 0 or patchSaliencyMap.shape != (patchSize, patchSize):
                    print(f"Error: Invalid saliency map for patch at index {patch_index} in WSI {wsi_name}.")
                    sys.exit(1)  # Exit the script immediately
                else:
                    patchImportance = np.mean(patchSaliencyMap)

                # Determine weight based on patch importance
                weight = patchImportance if patchImportance >= 0.25 else 0

                # Update the metadata DataFrame with the new values
                metadata.loc[(metadata['Sample'] == wsi_name) & (metadata['Index'] == patch_index), 'Densenet_Gradcam_Saliency_Importance'] = patchImportance
                metadata.loc[(metadata['Sample'] == wsi_name) & (metadata['Index'] == patch_index), 'Densenet_Gradcam_Weight'] = weight

                # Verify that the update was successful
                updated_row = metadata[(metadata['Sample'] == wsi_name) & (metadata['Index'] == patch_index)]
                if updated_row.empty or updated_row['Densenet_Gradcam_Saliency_Importance'].isnull().all() or updated_row['Densenet_Gradcam_Weight'].isnull().all():
                    print(f"Error: Failed to update metadata for patch at index {patch_index} in WSI {wsi_name}.")
                    sys.exit(1)  # Exit the script immediately

            
            
            # Visualize the Grad-CAM
            heatmap_pp, result_pp = visualize_cam(mask_pp_cpu, torch_img)

            # Stack the images together
            # Resize all images to 256x256
            resized_torch_img = F.interpolate(torch_img, size=(400, 400), mode='bilinear', align_corners=False).squeeze()
            resized_heatmap_pp = F.interpolate(heatmap_pp.unsqueeze(0), size=(400, 400), mode='bilinear', align_corners=False).squeeze()
            resized_result_pp = F.interpolate(result_pp.unsqueeze(0), size=(400, 400), mode='bilinear', align_corners=False).squeeze()

            # Stack the resized images horizontally
            concatenated_images = torch.cat([resized_torch_img, resized_heatmap_pp, resized_result_pp], dim=2)  # Concatenate along width

            # Save the concatenated image
            save_path = os.path.join(output_dir, f"{wsi_name_without_extension}_gradcam_combined.jpg")
            save_image(concatenated_images, save_path)


            # Save the upscaled result
            result_filename = os.path.join(output_dir, f"{wsi_name_without_extension}_gradcam_result.jpg")
            save_image(result_pp, result_filename)

            print(f"Saved Grad-CAM results for {wsi_name} to {save_path} and {result_filename}")
        else:
            print(f"Saliency map for {wsi_name_without_extension} not found at {saliency_map_filename}")
    else:
        print(f"WSI image for {wsi_name} not found at {wsi_image_filename}")
    
# Save the updated DataFrame to a new CSV file
output_csv_path = '/data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/CSV_Files/metadata_patches_with_grad_cam++_binary_label.csv'
metadata.to_csv(output_csv_path, index=False)
print(f"Updated metadata saved to {output_csv_path}.")    



Directory '/data/users4/pafshin1/My_Projects copy/Rands/DATA/PATCHES/INPUT_WSI' found. Proceeding with loading saliency maps.


Processing WSI Images:   0%|          | 0/66 [00:00<?, ?it/s]

Processing WSI: 10_3


Processing WSI Images:   2%|▏         | 1/66 [00:29<32:23, 29.89s/it]

Saved Grad-CAM results for 10_3 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/10_3_gradcam_combined.jpg and /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/10_3_gradcam_result.jpg
Processing WSI: 11_3


...

Saved Grad-CAM results for 7_2 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/7_2_gradcam_combined.jpg and /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/7_2_gradcam_result.jpg
Processing WSI: 8_1


Processing WSI Images:  98%|█████████▊| 65/66 [30:47<00:34, 34.38s/it]

Saved Grad-CAM results for 8_1 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/8_1_gradcam_combined.jpg and /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/8_1_gradcam_result.jpg
Processing WSI: 9_3


Processing WSI Images: 100%|██████████| 66/66 [30:59<00:00, 28.17s/it]

Saved Grad-CAM results for 9_3 to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/9_3_gradcam_combined.jpg and /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/Grad-Cam++/Results/1st_batch_heatmaps/9_3_gradcam_result.jpg
Updated metadata saved to /data/users4/pafshin1/Implementation/Vision Transformer/EMBC_Updated_Automatic_Thresholding_Implementation/CSV_Files/metadata_patches_with_grad_cam++_binary_label.csv.


In [18]:
#now a metadata with binary_label and grad-cam++scores which can be used for class dataset later in main.py
metadata

,Sample,Index,Row,Column,Binary_Label,Densenet_Gradcam_Saliency_Importance,Densenet_Gradcam_Weight
0,2_1,1,5600,2000,0,0.262619,0.262619
1,2_1,2,6000,2000,0,0.285723,0.285723
2,2_1,3,5200,2400,0,0.277677,0.277677
3,2_1,4,5600,2400,0,0.307524,0.307524
4,2_1,5,6000,2400,0,0.337250,0.337250
...,...,...,...,...,...,...,...
36123,70_1,1200,6800,19600,0,0.848264,0.848264
36124,70_1,1201,7200,19600,0,0.842641,0.842641
36125,70_1,1202,7600,19600,0,0.837018,0.837018
36126,70_1,1203,8000,19600,0,0.831395,0.831395
